# Конспект по ютуб видео (https://www.youtube.com/watch?v=kPxASj5wJBY)

## **Popularity based recommendatioanl system**

системы, которые предлагают пользователю продукты или услуги, которые наиболее популярны на местном или всемирном рынке на данной платформе

- Использует товары или услуги, которые популярны сейчас (в тренде)
- Ранжирует предметы основываясь на пользовательских рейтингах, истории продукта, кол-ву просмотров и так далее

In [3]:
import pandas as pd
import numpy as np

Загружаю датасет с фильмами

In [4]:
movies = pd.read_csv('movies.csv')
print(f'Колонки датасета: {movies.columns.tolist()}')
movies.head(3)

Колонки датасета: ['movieId', 'title', 'genres']


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


Загружаю датасет с рейтингом

In [5]:
ratings = pd.read_csv('ratings.csv')
print(f'Колонки датасета: {ratings.columns.tolist()}')
ratings.head(3)

Колонки датасета: ['userId', 'movieId', 'rating', 'timestamp']


,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182


Кол-во разных пользователей в датасете:

In [6]:
print(f'Кол-во разных пользователей в датасете: {len(ratings["userId"].value_counts().index.tolist())}')

Кол-во разных пользователей в датасете: 671


Объединяем два датасета по колонке с ID фильма

In [7]:
movie_data = pd.merge(ratings, movies, on='movieId')
movie_data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,31,2.5,1260759144,Dangerous Minds (1995),Drama
1,1,1029,3.0,1260759179,Dumbo (1941),Animation|Children|Drama|Musical
2,1,1061,3.0,1260759182,Sleepers (1996),Thriller
3,1,1129,2.0,1260759185,Escape from New York (1981),Action|Adventure|Sci-Fi|Thriller
4,1,1172,4.0,1260759205,Cinema Paradiso (Nuovo cinema Paradiso) (1989),Drama


Теперь, надо определить **параметры для Popularity Based Recomendational System**

Такими критериями *(параметрами)* будут:
- Фильмы с наибольшим рейтингом
- Кол-во просмотров

Средний рейтинг фильмов:

In [8]:
average_rate = movie_data['rating'].mean()
print(f'Средний рейтинг: {float(average_rate):.2f}')

Средний рейтинг: 3.54


Получаю средний рейтинг **для каждого фильма**

In [9]:
ag_ratings = movie_data.groupby('title')['rating'].mean()
ag_ratings.head()

title
"Great Performances" Cats (1998)           1.750000
$9.99 (2008)                               3.833333
'Hellboy': The Seeds of Creation (2004)    2.000000
'Neath the Arizona Skies (1934)            0.500000
'Round Midnight (1986)                     2.250000
Name: rating, dtype: float64

Сортировка фильмов по рейтингу

In [10]:
ag_ratings.sort_values(ascending=False).head(10)

title
Zerophilia (2005)                                                              5.0
'night Mother (1986)                                                           5.0
Zelary (2003)                                                                  5.0
Dorian Blues (2004)                                                            5.0
Disappearance of Haruhi Suzumiya, The (Suzumiya Haruhi no shôshitsu) (2010)    5.0
Two Ninas (1999)                                                               5.0
Seve (2014)                                                                    5.0
Dr. Jekyll and Mr. Hyde (1941)                                                 5.0
Two Escobars, The (2010)                                                       5.0
Drained (O cheiro do Ralo) (2006)                                              5.0
Name: rating, dtype: float64

Получаем кол-во оценок для каждого фильма

In [11]:
movie_data.groupby('title')['rating'].count().sort_values(ascending=False).head()

title
Forrest Gump (1994)                          341
Pulp Fiction (1994)                          324
Shawshank Redemption, The (1994)             311
Silence of the Lambs, The (1991)             304
Star Wars: Episode IV - A New Hope (1977)    291
Name: rating, dtype: int64

Создаю новый датафрейм с названием фильма, средним рейтингом и кол-вом оценок

In [12]:
final_data = pd.DataFrame(movie_data.groupby('title')['rating'].mean())
final_data['ratings_count'] = pd.DataFrame(movie_data.groupby('title')['rating'].count())
final_data.head()

,rating,ratings_count
title,,
"""Great Performances"" Cats (1998)",1.750000,2
$9.99 (2008),3.833333,3
'Hellboy': The Seeds of Creation (2004),2.000000,1
'Neath the Arizona Skies (1934),0.500000,1
'Round Midnight (1986),2.250000,2


Оставляю только те фильмы, у которых средний рейтинг **больше 4**, и кол-во оценок **не меньше 100**

In [13]:
final_data = final_data[(final_data['rating'] > 4) & (final_data['ratings_count'] > 100)]
final_data.head()

,rating,ratings_count
title,,
"Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)",4.096000,125
American Beauty (1999),4.236364,220
American History X (1998),4.023364,107
Apocalypse Now (1979),4.138393,112
Back to the Future (1985),4.015487,226


Округляю рейтинги

In [14]:
final_data['rating'] = round(final_data['rating'], 1)

Вывожу первые 5 фильмов, которые и **будут результатом** вывода пользователю с использованием `popularity based recommendational system`

In [15]:
final_data.sort_values(by='rating', ascending=False).head()


,rating,ratings_count
title,,
"Godfather, The (1972)",4.5,200
"Shawshank Redemption, The (1994)",4.5,311
"Godfather: Part II, The (1974)",4.4,135
"Usual Suspects, The (1995)",4.4,201
Pulp Fiction (1994),4.3,324


## Content-based Filltering 

Фильтрация на основе контента, собственно работает с **схожим контентом**
- Использует продукты и\или услуги которые похожи друг на друга
- Рекомендыет продукт **пользователю А** основываясь на оценке, который **пользователь А** дал продукту прошлый раз
- Не использует данные от других пользователей
- Использует техники по типу **Косинусного Сходства** для определения близости предметов
- Может рекомендовать пользователю один и тот же продукт раз за разом, даже если пользователь в нем не нуждается

--------------------------------

**Евклидово расстояние** - вычисляет `расстояние` между векторами

$$d(p,q) = \sqrt{\sum_{i=1}^n (q_i - p_i)^2}$$

где:
- $p,q$ - две точки в n-мерном евклидовом пространстве
- $p_i, q_i$ - векторы, ведущие из начала координат евклидова пространства (исходной точки)
- $n$ = n-мерное пространство

--------------------------------------------

**Косинусное сходство** *(расстояние)* - вычисляет `сходство` между векторами

$$similarity = cos(\theta) = \frac{A * B}{||A|| ||B||} = \frac{\sum_{i=1}^n A_i * B_i}{\sqrt{\sum_{i=1}^n(A_i)^2} * \sqrt{\sum_{i=1}^n(B_i)^2}}$$

где:
- $A * B$ = скалярное произведение векторов 
- $||A|| ||B||$ = произведение длин векторов A и B соответственно

---------------------

Реализация формулы косинусного сходства на Python

*Функция для вычисления длины вектора*

In [109]:
def vector_length(x: list | np.ndarray) -> float:
    '''Ф-я для вычисления скалярного умножения двух векторов'''
    
    return (sum([i * i for i in x]))**0.5

*Функция для вычисления косинусного сходства*

In [110]:
def cosine_similarity(x: list | np.ndarray, y: list | np.ndarray) -> float:
    if len(x) != len(y):
        raise TypeError('Вектора должны быть одинаковой длины')

    scalar_multiplication = sum([x[i]*y[i] for i in range(len(y))])
    return scalar_multiplication / (vector_length(x) * vector_length(y))


Генерация случайных векторов с размерностью dim (10)

In [113]:
dim = 10
x = np.random.rand(dim)
y = np.random.rand(dim)
print(x)
print(y)

[0.90724652 0.19590491 0.07446957 0.83833807 0.98366584 0.09320059
 0.71091866 0.84064528 0.31554865 0.68696659]
[0.0389287  0.32966902 0.33155936 0.64234272 0.84823222 0.14683138
 0.15395448 0.32164209 0.78248909 0.86125068]


Вывод сходства векторов

In [114]:
cosine_similarity(x, y)


np.float64(0.7763557708341264)

Как создать `Content-Based Recommendation System`

- Конвертировать базу данных в вектора (например используя эмбеддинги или кодирование)
- Искать косинусное сходство между запросом пользователя и всеми остальными продуктами